#### DATASET 'seguros'
- nome_contratante, str
- nr_documento_contratante, str
- cidade_contratante, str
- estado_contratante, str
- numero_apolice, int
- data_contratacao, date
- valor_pagamento, float
- valor_premio, float
- nome_beneficiario, str
- status_apolice, str
- Cobertura1, str
- Valor Cob 1, float
- Cobertura2, str
- Valor Cob 2, float
- Cobertura3, str
- Valor Cob 3, float
- capital_segurado, float

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, when, count, mean, stddev, to_date, lit, trim, round
from pyspark.sql.types import IntegerType, DoubleType, DateType, LongType, StringType
from functools import reduce

spark = SparkSession.builder.appName("Limpeza").getOrCreate()
caminho = "/Workspace/Users/actc@cesar.school/Grupo7-Setor-de-Seguros/data/processed/bronze/seguros_vida_encoding.csv"

df = spark.read\
    .format("csv")\
    .option("header", "true")\
    .option("sep", ";")\
    .load(caminho)

print(df.columns)
df.printSchema()
df.show()

### Limpeza inicial
- casting, trim, duplicatas, valores anormais, nulos

In [0]:
for c in df.columns:
    df = df.withColumn(c, trim(regexp_replace(col(c), " +", " ")))

cols_dinheiro = [
    "valor_pagamento", "valor_premio", "Valor Cob 1", 
    "Valor Cob 2", "Valor Cob 3", "capital_segurado"
]
cols_int = ["numero_apolice"]
cols_date = ["data_contratacao"]

for c in cols_dinheiro:
    df = df.withColumn(c, regexp_replace(col(c), ",", ".").cast(DoubleType()))

for c in cols_int:
    df = df.withColumn(c, round(col(c).cast(LongType()), 2))

for c in cols_date:
    clean_col = trim(col(c))
    
    df = df.withColumn(c,to_date(clean_col, "dd/MM/yyyy"))

Checamos se há valores monetários negativos

In [0]:
conditions = [col(c) < 0 for c in cols_dinheiro]
combined_condition = reduce(lambda a, b: a | b, conditions)

negative_rows = df.filter(combined_condition)
negative_count = negative_rows.count()

print(f"Encontrou {negative_count} linhas com valores negativos.") # Não encontramos
if negative_count > 0:
    display(negative_rows)

Checamos se há valores discrepantes (+- 4 desvios padrão da média) e removemos

In [0]:
for c in cols_dinheiro:
    # Média e desvio padrão
    stats = df.select(mean(col(c)).alias("avg"), stddev(col(c)).alias("std")).collect()[0]
    mean_val = stats["avg"]
    std_val = stats["std"]
    
    if mean_val is not None and std_val is not None:
        lower_bound = max(0, mean_val - (4 * std_val))
        upper_bound = mean_val + (4 * std_val)
        
        outliers = df.filter((col(c) < lower_bound) | (col(c) > upper_bound))
        count_outliers = outliers.count()
        
        if count_outliers > 0:
            print(f"Coluna '{c}': {count_outliers} outliers (De {lower_bound:.2f} até {upper_bound:.2f})")
            display(outliers)
            df = df.filter((col(c) >= lower_bound) & (col(c) <= upper_bound))
    else:
        print("Erro ao calcular medidas.")

Checa se há linhas duplicadas

In [0]:
total_count = df.count()
distinct_count = df.distinct().count()
duplicate_count = total_count - distinct_count

print(f"Total: {total_count}")
print(f"Distintas: {distinct_count}")
print(f"Duplicatas: {duplicate_count}")

if duplicate_count > 0:
    df.groupby(df.columns).count().filter("count > 1").show()

Checamos se há dados nulos ou vazios

In [0]:
null_check_exprs = []
for c in df.columns:
    if isinstance(df.schema[c].dataType, StringType):
        null_check_exprs.append(count(when(col(c).isNull() | (col(c) == ""), c)).alias(c))
    else:
        null_check_exprs.append(count(when(col(c).isNull(), c)).alias(c))

null_counts = df.select(null_check_exprs)
display(null_counts)

In [0]:
# Transforma strings vazias em NULL e remove linhas
for c in df.columns:
    if isinstance(df.schema[c].dataType, StringType):
        df = df.withColumn(c, when(col(c) == "", None).otherwise(col(c)))

df = df.dropna(how='any')

Checando datas mais antigas que uma década

In [0]:
def check_date(dataframe, date_col, min_date, max_date):
    out_of_bounds = dataframe.filter(
        (col(date_col) < to_date(lit(min_date), "dd/MM/yyyy")) | 
        (col(date_col) > to_date(lit(max_date), "dd/MM/yyyy"))
    )
    
    cnt = out_of_bounds.count()
    print(f"Coluna '{date_col}': {cnt} linhas fora do intervalo {min_date} - {max_date}")
    if cnt > 0:
        display(out_of_bounds)

start_threshold = "01/01/2015"
start_threshold = "31/12/2025"
check_date(df, "data_contratacao", start_threshold, start_threshold)

In [0]:
display(df)